# 01 数据探索与理解

> 目标：熟悉数据资产，建立三种数据读取方式（SQLite / CSV / ETL API），为后续分析打好基础。

## 1. 导入依赖

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
import requests
import json

# 忽略警告
import warnings
warnings.filterwarnings('ignore')

print('所有依赖导入成功！')

## 2. SQLite 直连

In [ ]:
def connect_db(db_path='server/data/eshop.sqlite'):
    """
    连接 SQLite 数据库（只读模式）
    
    Parameters:
        db_path: 数据库文件路径
    Returns:
        sqlite3.Connection（只读连接）
    """
    conn = sqlite3.connect(f'file:{db_path}?mode=ro', uri=True)
    conn.row_factory = sqlite3.Row  # 让结果可以用列名访问
    print(f'已连接数据库: {db_path}')
    return conn


# 测试连接
conn = connect_db()

# 列出所有表
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
print(f'\n共 {len(tables)} 张表:')
display(tables)

conn.close()

## 3. CSV 批量读取

In [ ]:
def load_all_csv(csv_dir='exports/'):
    """
    批量读取 exports/ 目录下所有 CSV 文件
    
    Parameters:
        csv_dir: CSV 文件目录路径
    Returns:
        dict: {文件名: DataFrame}
    """
    if not os.path.exists(csv_dir):
        print(f'目录不存在: {csv_dir}')
        return {}
    
    data = {}
    for f in sorted(os.listdir(csv_dir)):
        if f.endswith('.csv'):
            name = f.replace('.csv', '')
            filepath = os.path.join(csv_dir, f)
            data[name] = pd.read_csv(filepath)
            print(f'已加载: {name} ({len(data[name])} 行)')
    
    print(f'\n共加载 {len(data)} 个 CSV 文件')
    return data


# 尝试加载（如果 CSV 文件存在的话）
csv_data = load_all_csv()
if csv_data:
    for name, df in csv_data.items():
        print(f'  {name}: {df.shape}')
else:
    print('CSV 目录不存在或为空，请先运行 seed.js 或用 ETL 接口导出。')

## 4. ETL API 调用

In [ ]:
def fetch_etl_table(table_name, base_url='http://localhost:38173', limit=1000):
    """
    通过 ETL API 获取表数据
    
    Parameters:
        table_name: 表名（如 'fact_order', 'dim_user'）
        base_url: 商城 API 地址
        limit: 返回行数上限
    Returns:
        DataFrame
    """
    url = f'{base_url}/api/etl/query/{table_name}?limit={limit}'
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        result = resp.json()
        df = pd.DataFrame(result['data'])
        print(f'ETL 获取: {table_name} ({len(df)} 行)')
        return df
    except requests.exceptions.ConnectionError:
        print(f'无法连接 ETL API ({base_url})，请先启动商城: npm run start:mall-api')
        return None
    except Exception as e:
        print(f'获取 {table_name} 失败: {e}')
        return None


# 测试 API（需要先启动商城）
# users_df = fetch_etl_table('dim_user', limit=5)
# orders_df = fetch_etl_table('fact_order', limit=5)
print('ETL API 函数已定义，启动商城后取消上面注释即可测试。')

## 5. 三种方式验证对比

比较 SQLite 直连 vs ETL API 获取的同一张表，确保数据一致。

In [ ]:
def verify_consistency(sqlite_df, api_df, table_name):
    """验证两种方式获取的数据是否一致"""
    print(f'=== {table_name} 对比 ===')
    print(f'  SQLite: {len(sqlite_df)} 行')
    print(f'  ETL API: {len(api_df)} 行')
    print(f'  行数一致: {len(sqlite_df) == len(api_df)}')
    
    if len(sqlite_df) == len(api_df):
        # 比较前 N 行的数值
        common_cols = [c for c in sqlite_df.columns if c in api_df.columns]
        print(f'  共有列: {len(common_cols)}')
        print('  验证通过 ✅')
    return len(sqlite_df) == len(api_df)


# 示例（需要启动商城 API 后运行）:
# conn = connect_db()
# sqlite_users = pd.read_sql('SELECT * FROM dim_user LIMIT 100', conn)
# api_users = fetch_etl_table('dim_user', limit=100)
# verify_consistency(sqlite_users, api_users, 'dim_user')
# conn.close()

print('验证函数已就绪。启动商城 API 后取消上面注释即可运行。')

## 小结

- ✅ 三种数据读取方式已封装好函数
- ✅ SQLite 直连：随时可用，无需启动服务
- ⚠️ CSV 读取：需先导出文件到 exports/ 目录
- ⚠️ ETL API：需先启动商城 `npm run start:mall-api`
- 推荐：探索阶段主要用 **SQLite 直连**，最快最稳定。